# Task 1: Product Label Prediction
## Classical vectorization comparison + DistilBERT multi-label benchmark

This notebook builds a **text-level multi-label product prediction** dataset for:

- flower
- oil
- gummies
- vape
- topical

It uses:

- `pair_level_gpt41mini_clean.csv` for pair-level labels
- `cleaned_dataset.csv` for cleaned source text

Then it:

1. merges pair-level labels back to cleaned text
2. removes emojis for Task 1
3. builds the text-level 5-label dataset
4. compares classical vectorization/model pipelines
5. tunes thresholds on the validation set
6. trains a DistilBERT multi-label benchmark
7. compares final performance on the test set

In [62]:
# =========================
# 1. Imports
# =========================
import os
import re
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, HashingVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    hamming_loss, classification_report
)
from gensim.models import Word2Vec

warnings.filterwarnings("ignore")

In [63]:
# =========================
# 2. Paths
# =========================
PROJECT_ROOT = Path("/Users/xiyuehuang/Desktop/CBB 750: BIS 550/Cannabis-Analysis")
PAIR_PATH = PROJECT_ROOT / "data_processing" / "data" / "final" / "pair_level_gpt41mini_clean.csv"
CLEAN_PATH = PROJECT_ROOT / "data_processing" / "data" / "processed" / "cleaned_dataset.csv"
OUTPUT_DIR = PROJECT_ROOT / "Modeling" / "product_label_prediction_outputs_noemoji"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PAIR_PATH exists:", PAIR_PATH.exists(), PAIR_PATH)
print("CLEAN_PATH exists:", CLEAN_PATH.exists(), CLEAN_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

PAIR_PATH exists: True /Users/xiyuehuang/Desktop/CBB 750: BIS 550/Cannabis-Analysis/data_processing/data/final/pair_level_gpt41mini_clean.csv
CLEAN_PATH exists: True /Users/xiyuehuang/Desktop/CBB 750: BIS 550/Cannabis-Analysis/data_processing/data/processed/cleaned_dataset.csv
OUTPUT_DIR: /Users/xiyuehuang/Desktop/CBB 750: BIS 550/Cannabis-Analysis/Modeling/product_label_prediction_outputs_noemoji


In [64]:
# =========================
# 3. Load data
# =========================
pair_df = pd.read_csv(PAIR_PATH)
clean_df = pd.read_csv(CLEAN_PATH)

print("pair_df shape:", pair_df.shape)
print("clean_df shape:", clean_df.shape)
print("\npair_df columns:", pair_df.columns.tolist())
print("\nclean_df columns:", clean_df.columns.tolist())

pair_df shape: (7031, 4)
clean_df shape: (110592, 35)

pair_df columns: ['text_id', 'product', 'sentiment', 'date_utc']

clean_df columns: ['text_id', 'id', 'source', 'subreddit', 'author', 'score', 'created_utc', 'date_utc', 'year', 'month', 'searched_subreddit', 'searched_keyword', 'text', 'text_length_chars', 'text_length_tokens', 'is_deleted_or_empty', 'product_labels_weak', 'product_labels_weak_str', 'product_flower_weak', 'product_oil_weak', 'product_gummies_weak', 'product_vape_weak', 'product_topical_weak', 'product_label_canonical', 'product_label_source', 'sentiment_label_weak', 'sentiment_label_source', 'annotation_batch', 'annotation_prompt_version', 'product_label_gold', 'product_label_gold_source', 'product_label_notes', 'sentiment_label_gold', 'sentiment_label_gold_source', 'sentiment_label_notes']


In [65]:
# =========================
# 4. Helper functions
# =========================
def find_col(df, candidates, required=True):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise KeyError(f"Could not find any of columns: {candidates}")
    return None

def basic_clean_text(text):
    text = "" if pd.isna(text) else str(text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Remove emoji and most pictographs/symbols for Task 1
_EMOJI_PATTERN = re.compile(
    "["                               
    "\U0001F600-\U0001F64F"         # emoticons
    "\U0001F300-\U0001F5FF"         # symbols & pictographs
    "\U0001F680-\U0001F6FF"         # transport & map
    "\U0001F1E0-\U0001F1FF"         # flags
    "\U00002702-\U000027B0"
    "\U000024C2-\U0001F251"
    "\U0001F900-\U0001F9FF"
    "\U0001FA70-\U0001FAFF"
    "\U00002600-\U000026FF"
    "\U00002300-\U000023FF"
    "]+", flags=re.UNICODE
)

def remove_emoji(text):
    text = "" if pd.isna(text) else str(text)
    text = _EMOJI_PATTERN.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_id(s):
    s = "" if pd.isna(s) else str(s)
    return s.strip()

def tokenize_for_w2v(text):
    text = str(text).lower()
    return re.findall(r"\b[a-zA-Z][a-zA-Z0-9_-]*\b", text)

def avg_w2v(tokens, model, dim):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if len(vecs) == 0:
        return np.zeros(dim, dtype=float)
    return np.mean(vecs, axis=0)

def multilabel_metrics(y_true, y_pred, label_names, model_name):
    summary = {
        "model": model_name,
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "subset_accuracy": accuracy_score(y_true, y_pred),
    }
    per_label = pd.DataFrame({
        "label": label_names,
        "precision": precision_score(y_true, y_pred, average=None, zero_division=0),
        "recall": recall_score(y_true, y_pred, average=None, zero_division=0),
        "f1": f1_score(y_true, y_pred, average=None, zero_division=0),
    })
    return summary, per_label

def tune_thresholds(y_true, y_score, label_names, grid=None):
    if grid is None:
        grid = np.arange(0.10, 0.91, 0.05)
    best_thresholds = []
    for j in range(y_true.shape[1]):
        best_t = 0.5
        best_f1 = -1
        for t in grid:
            pred = (y_score[:, j] >= t).astype(int)
            f1 = f1_score(y_true[:, j], pred, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = float(t)
        best_thresholds.append(best_t)
    return np.array(best_thresholds)

def apply_thresholds(y_score, thresholds):
    return (y_score >= thresholds.reshape(1, -1)).astype(int)


In [66]:
# =========================
# 5. Identify columns
# =========================
pair_id_col = find_col(pair_df, ["text_id", "ext_id", "id"])
product_col = find_col(pair_df, ["product", "product_label", "product_type"])
sentiment_col = find_col(pair_df, ["sentiment"], required=False)

clean_id_col = find_col(clean_df, ["text_id", "id", "ext_id", "record_id", "doc_id"])
clean_text_col = find_col(clean_df, ["text", "clean_text", "full_text", "body", "content"])

print("pair_id_col   =", pair_id_col)
print("product_col   =", product_col)
print("sentiment_col =", sentiment_col)
print("clean_id_col  =", clean_id_col)
print("clean_text_col=", clean_text_col)


pair_id_col   = text_id
product_col   = product
sentiment_col = sentiment
clean_id_col  = text_id
clean_text_col= text


In [67]:
# =========================
# 6. Merge pair-level labels back to cleaned text
# =========================
pair_small = pair_df[[pair_id_col, product_col] + ([sentiment_col] if sentiment_col else [])].copy()
pair_small = pair_small.rename(columns={pair_id_col: "text_id", product_col: "product"})
pair_small["text_id"] = pair_small["text_id"].map(normalize_id)

clean_small = clean_df[[clean_id_col, clean_text_col]].copy()
clean_small = clean_small.rename(columns={clean_id_col: "merge_id", clean_text_col: "text"})
clean_small["merge_id"] = clean_small["merge_id"].map(normalize_id)
clean_small["text"] = clean_small["text"].map(basic_clean_text)

merged_df = pair_small.merge(clean_small, left_on="text_id", right_on="merge_id", how="left")

print("merged_df shape:", merged_df.shape)
print("Rows missing text after merge:", merged_df["text"].isna().sum())
merged_df.head()

merged_df shape: (7031, 5)
Rows missing text after merge: 0


,text_id,product,sentiment,merge_id,text
0,hqrt7lw,vape,positive,hqrt7lw,Blueberry is and pancake are very well balance...
1,hqs1aeh,vape,neutral,hqs1aeh,"The deeper the oil gets into your lungs, the h..."
2,hqs1aeh,oil,neutral,hqs1aeh,"The deeper the oil gets into your lungs, the h..."
3,hqs7u29,oil,neutral,hqs7u29,thank you for the advice! i have tried the cbd...
4,hqs7u29,flower,neutral,hqs7u29,thank you for the advice! i have tried the cbd...


In [68]:
# =========================
# 7. Remove emojis for Task 1 product prediction
# =========================
merged_df["text"] = merged_df["text"].fillna("").astype(str)
merged_df["text_noemoji"] = merged_df["text"].map(remove_emoji).map(basic_clean_text)

print("Example before/after:")
sample_idx = merged_df["text"].str.len().sort_values(ascending=False).index[:3]
for idx in sample_idx:
    print("\n--- ORIGINAL ---")
    print(merged_df.loc[idx, "text"][:300])
    print("--- NO EMOJI ---")
    print(merged_df.loc[idx, "text_noemoji"][:300])

Example before/after:

--- ORIGINAL ---
No problem at all. I’m going to assume that the reason vaping was suggested to you, is due to the health concerns of smoking. Some states until recently, like New York, didn’t even allow you to buy flower. You could only get the oil. That’s changed now, but it’s funny how the states and the doctors 
--- NO EMOJI ---
No problem at all. I’m going to assume that the reason vaping was suggested to you, is due to the health concerns of smoking. Some states until recently, like New York, didn’t even allow you to buy flower. You could only get the oil. That’s changed now, but it’s funny how the states and the doctors 

--- ORIGINAL ---
No problem at all. I’m going to assume that the reason vaping was suggested to you, is due to the health concerns of smoking. Some states until recently, like New York, didn’t even allow you to buy flower. You could only get the oil. That’s changed now, but it’s funny how the states and the doctors 
--- NO EMOJI ---
No pr

In [69]:
# =========================
# 8. Build text-level 5-label dataset
# =========================
FINAL_PRODUCTS = ["flower", "oil", "gummies", "vape", "topical"]

work_df = merged_df.copy()
work_df["product"] = work_df["product"].astype(str).str.lower().str.strip()
work_df = work_df[work_df["product"].isin(FINAL_PRODUCTS)].copy()
work_df = work_df[work_df["text_noemoji"].notna()].copy()
work_df["text_noemoji"] = work_df["text_noemoji"].astype(str).str.strip()
work_df = work_df[work_df["text_noemoji"] != ""].copy()

text_level = (
    work_df.assign(value=1)
    .pivot_table(
        index=["text_id", "text_noemoji"],
        columns="product",
        values="value",
        aggfunc="max",
        fill_value=0
    )
    .reset_index()
)

for p in FINAL_PRODUCTS:
    if p not in text_level.columns:
        text_level[p] = 0

text_level = text_level.rename(columns={"text_noemoji": "text"})
text_level = text_level[["text_id", "text"] + FINAL_PRODUCTS].copy()

print("text_level shape:", text_level.shape)
print(text_level[FINAL_PRODUCTS].sum())
text_level.head()

text_level shape: (4401, 7)
product
flower     1557
oil        1789
gummies    1796
vape       1584
topical     219
dtype: int64


product,text_id,text,flower,oil,gummies,vape,topical
0,hqrt7lw,Blueberry is and pancake are very well balance...,0,0,0,1,0
1,hqs1aeh,"The deeper the oil gets into your lungs, the h...",0,1,0,1,0
2,hqs7u29,thank you for the advice! i have tried the cbd...,1,1,0,0,0
3,hqsgolp,Happy new years! Wish edibles hit me i’ve trie...,0,1,1,1,0
4,hqt14xp,For me it's both. Oil worked acutely from the ...,0,1,0,1,0


In [70]:
# =========================
# 9. Save text-level dataset
# =========================
text_level_path = OUTPUT_DIR / "text_level_5label_noemoji.csv"
text_level.to_csv(text_level_path, index=False)
print("Saved:", text_level_path)

Saved: /Users/xiyuehuang/Desktop/CBB 750: BIS 550/Cannabis-Analysis/Modeling/product_label_prediction_outputs_noemoji/text_level_5label_noemoji.csv


In [71]:
# =========================
# 10. Train / val / test split by text_id
# =========================
train_df, test_df = train_test_split(text_level, test_size=0.15, random_state=SEED)
train_df, val_df = train_test_split(train_df, test_size=0.1764705882, random_state=SEED)  # 0.15 val overall

label_names = FINAL_PRODUCTS

X_train = train_df["text"].tolist()
X_val = val_df["text"].tolist()
X_test = test_df["text"].tolist()

y_train = train_df[label_names].values.astype(int)
y_val = val_df[label_names].values.astype(int)
y_test = test_df[label_names].values.astype(int)

print("Train:", len(X_train), y_train.shape)
print("Val:  ", len(X_val), y_val.shape)
print("Test: ", len(X_test), y_test.shape)

Train: 3080 (3080, 5)
Val:   660 (660, 5)
Test:  661 (661, 5)


## Classical models

Below we compare several **non-transformer** vectorization/model setups.  
Each one is threshold-tuned on the validation set before final evaluation on the test set.

In [72]:
# =========================
# 11. CountVectorizer + LinearSVC
# =========================
count_model = Pipeline([
    ("vec", CountVectorizer(
        lowercase=True,
        strip_accents="unicode",
        min_df=2,
        max_df=0.98,
        ngram_range=(1, 2)
    )),
    ("clf", OneVsRestClassifier(LinearSVC(class_weight="balanced")))
])

count_model.fit(X_train, y_train)

# Use decision_function for threshold tuning
count_val_score = np.column_stack([
    est.decision_function(count_model.named_steps["vec"].transform(X_val))
    for est in count_model.named_steps["clf"].estimators_
])
count_test_score = np.column_stack([
    est.decision_function(count_model.named_steps["vec"].transform(X_test))
    for est in count_model.named_steps["clf"].estimators_
])

count_thresholds = tune_thresholds(y_val, count_val_score, label_names)
count_pred = apply_thresholds(count_test_score, count_thresholds)

count_summary, count_per_label = multilabel_metrics(y_test, count_pred, label_names, "Count + LinearSVC")
count_summary, count_per_label


({'model': 'Count + LinearSVC',
  'micro_f1': 0.8845208845208845,
  'macro_f1': 0.8460177934704222,
  'weighted_f1': 0.8830023240951503,
  'micro_precision': 0.9230769230769231,
  'micro_recall': 0.8490566037735849,
  'hamming_loss': 0.07110438729198185,
  'subset_accuracy': 0.7019667170953101},
      label  precision    recall        f1
 0   flower   0.839286  0.780083  0.808602
 1      oil   0.928270  0.883534  0.905350
 2  gummies   0.951128  0.913357  0.931860
 3     vape   0.961039  0.853846  0.904277
 4  topical   1.000000  0.515152  0.680000)

In [73]:
# =========================
# 12. HashingVectorizer + LinearSVC
# =========================
hash_model = Pipeline([
    ("vec", HashingVectorizer(
        lowercase=True,
        strip_accents="unicode",
        n_features=2**18,
        alternate_sign=False,
        ngram_range=(1, 2)
    )),
    ("clf", OneVsRestClassifier(LinearSVC(class_weight="balanced")))
])

hash_model.fit(X_train, y_train)

hash_val_score = np.column_stack([
    est.decision_function(hash_model.named_steps["vec"].transform(X_val))
    for est in hash_model.named_steps["clf"].estimators_
])
hash_test_score = np.column_stack([
    est.decision_function(hash_model.named_steps["vec"].transform(X_test))
    for est in hash_model.named_steps["clf"].estimators_
])

hash_thresholds = tune_thresholds(y_val, hash_val_score, label_names)
hash_pred = apply_thresholds(hash_test_score, hash_thresholds)

hash_summary, hash_per_label = multilabel_metrics(y_test, hash_pred, label_names, "Hashing + LinearSVC")
hash_summary, hash_per_label


({'model': 'Hashing + LinearSVC',
  'micro_f1': 0.8642602948652771,
  'macro_f1': 0.8126445539448719,
  'weighted_f1': 0.8623519785689113,
  'micro_precision': 0.9371554575523704,
  'micro_recall': 0.8018867924528302,
  'hamming_loss': 0.08078668683812405,
  'subset_accuracy': 0.6611195158850227},
      label  precision    recall        f1
 0   flower   0.856502  0.792531  0.823276
 1      oil   0.958333  0.831325  0.890323
 2  gummies   0.967611  0.862816  0.912214
 3     vape   0.966019  0.765385  0.854077
 4  topical   0.933333  0.424242  0.583333)

In [74]:
# =========================
# 13. Character n-grams + Logistic Regression
# =========================
char_model = Pipeline([
    ("vec", CountVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_df=0.99
    )),
    ("clf", OneVsRestClassifier(
        LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear")
    ))
])

char_model.fit(X_train, y_train)

char_val_score = np.column_stack([
    est.predict_proba(char_model.named_steps["vec"].transform(X_val))[:, 1]
    for est in char_model.named_steps["clf"].estimators_
])
char_test_score = np.column_stack([
    est.predict_proba(char_model.named_steps["vec"].transform(X_test))[:, 1]
    for est in char_model.named_steps["clf"].estimators_
])

char_thresholds = tune_thresholds(y_val, char_val_score, label_names)
char_pred = apply_thresholds(char_test_score, char_thresholds)

char_summary, char_per_label = multilabel_metrics(y_test, char_pred, label_names, "Char n-gram + LogReg")
char_summary, char_per_label

({'model': 'Char n-gram + LogReg',
  'micro_f1': 0.8855911919578746,
  'macro_f1': 0.8670222936304892,
  'weighted_f1': 0.8857297981097948,
  'micro_precision': 0.8989310009718173,
  'micro_recall': 0.8726415094339622,
  'hamming_loss': 0.07231467473524962,
  'subset_accuracy': 0.7034795763993948},
      label  precision    recall        f1
 0   flower   0.806584  0.813278  0.809917
 1      oil   0.939394  0.871486  0.904167
 2  gummies   0.930403  0.916968  0.923636
 3     vape   0.921260  0.900000  0.910506
 4  topical   0.857143  0.727273  0.786885)

In [75]:
# =========================
# 14. Word2Vec average embeddings + Logistic Regression
# =========================
X_train_tok = [tokenize_for_w2v(t) for t in X_train]
X_val_tok   = [tokenize_for_w2v(t) for t in X_val]
X_test_tok  = [tokenize_for_w2v(t) for t in X_test]

W2V_DIM = 100
w2v = Word2Vec(
    sentences=X_train_tok,
    vector_size=W2V_DIM,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=10,
    seed=SEED
)

X_train_w2v = np.vstack([avg_w2v(toks, w2v, W2V_DIM) for toks in X_train_tok])
X_val_w2v   = np.vstack([avg_w2v(toks, w2v, W2V_DIM) for toks in X_val_tok])
X_test_w2v  = np.vstack([avg_w2v(toks, w2v, W2V_DIM) for toks in X_test_tok])

w2v_model = OneVsRestClassifier(
    LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear")
)
w2v_model.fit(X_train_w2v, y_train)

w2v_val_score = np.column_stack([
    est.predict_proba(X_val_w2v)[:, 1] for est in w2v_model.estimators_
])
w2v_test_score = np.column_stack([
    est.predict_proba(X_test_w2v)[:, 1] for est in w2v_model.estimators_
])

w2v_thresholds = tune_thresholds(y_val, w2v_val_score, label_names)
w2v_pred = apply_thresholds(w2v_test_score, w2v_thresholds)

w2v_summary, w2v_per_label = multilabel_metrics(y_test, w2v_pred, label_names, "Word2Vec avg + LogReg")
w2v_summary, w2v_per_label

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


({'model': 'Word2Vec avg + LogReg',
  'micro_f1': 0.7399366802351877,
  'macro_f1': 0.665689650279391,
  'weighted_f1': 0.7414770084995922,
  'micro_precision': 0.7106863596872285,
  'micro_recall': 0.7716981132075472,
  'hamming_loss': 0.17397881996974282,
  'subset_accuracy': 0.4069591527987897},
      label  precision    recall        f1
 0   flower   0.612426  0.858921  0.715026
 1      oil   0.738095  0.746988  0.742515
 2  gummies   0.795082  0.700361  0.744722
 3     vape   0.788530  0.846154  0.816327
 4  topical   0.289474  0.333333  0.309859)

In [76]:
# =========================
# 15. Classical comparison table
# =========================
classical_results = pd.DataFrame([
    count_summary, hash_summary, char_summary, w2v_summary
]).sort_values("macro_f1", ascending=False)

classical_results

,model,micro_f1,macro_f1,weighted_f1,micro_precision,micro_recall,hamming_loss,subset_accuracy
2,Char n-gram + LogReg,0.885591,0.867022,0.885730,0.898931,0.872642,0.072315,0.703480
0,Count + LinearSVC,0.884521,0.846018,0.883002,0.923077,0.849057,0.071104,0.701967
1,Hashing + LinearSVC,0.864260,0.812645,0.862352,0.937155,0.801887,0.080787,0.661120
3,Word2Vec avg + LogReg,0.739937,0.665690,0.741477,0.710686,0.771698,0.173979,0.406959


## DistilBERT multi-label benchmark

This section adds a transformer model **after** the classical comparison.

The model:
- takes raw text as input
- predicts 5 labels with **sigmoid** outputs
- uses **BCEWithLogitsLoss**
- tunes one threshold per label on the validation set

In [ ]:
# =========================
# 16. DistilBERT multi-label
# =========================

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW   # <-- from torch, NOT transformers
from tqdm import tqdm

MODEL_NAME = "distilbert-base-uncased"
LABEL_COLS = ["flower", "oil", "gummies", "vape", "topical"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# Dataset
# -------------------------
class TextDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.texts = df["text"].tolist()
        self.labels = df[LABEL_COLS].values.astype(np.float32)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

# -------------------------
# Model
# -------------------------
class DistilBERTMultiLabel(nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]  # CLS token
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        return logits

# -------------------------
# Prepare data
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = TextDataset(train_df, tokenizer)
val_ds   = TextDataset(val_df, tokenizer)
test_ds  = TextDataset(test_df, tokenizer)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=8)
test_loader  = DataLoader(test_ds, batch_size=8)

# -------------------------
# Model + optimizer
# -------------------------
model = DistilBERTMultiLabel(len(LABEL_COLS)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=2e-5)

# -------------------------
# Training loop
# -------------------------
EPOCHS = 3

def evaluate(loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask)
            probs = torch.sigmoid(logits)

            all_preds.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    return np.vstack(all_preds), np.vstack(all_labels)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    val_probs, val_labels = evaluate(val_loader)
    val_preds = (val_probs > 0.5).astype(int)

    print("Epoch", epoch+1,
          "Loss:", total_loss/len(train_loader),
          "Macro F1:", f1_score(val_labels, val_preds, average="macro"))

# -------------------------
# Threshold tuning
# -------------------------
val_probs, val_labels = evaluate(val_loader)

thresholds = {}
for i, label in enumerate(LABEL_COLS):
    best_t = 0.5
    best_f1 = 0
    for t in np.arange(0.1, 0.9, 0.05):
        preds = (val_probs[:, i] > t).astype(int)
        f1 = f1_score(val_labels[:, i], preds)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    thresholds[label] = best_t

print("Best thresholds:", thresholds)

# -------------------------
# Test evaluation
# -------------------------
test_probs, test_labels = evaluate(test_loader)

test_preds = np.zeros_like(test_probs)
for i, label in enumerate(LABEL_COLS):
    test_preds[:, i] = (test_probs[:, i] > thresholds[label]).astype(int)

print("Final Macro F1:", f1_score(test_labels, test_preds, average="macro"))

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 